# Vòng tối ưu 2 — Sweep cấu hình + kiểm tra ổn định (Colab T4)

Notebook này tìm cấu hình tốt nhất một cách **có kỷ luật, không rò rỉ dữ liệu**:

1. **Sweep 8 cấu hình** (~70–90 phút): mỗi config train xong sẽ ghi lại metrics. Cấu hình thắng được chọn theo **validation ROC-AUC** — tuyệt đối không chọn theo test, vì chọn config theo test = gian lận số liệu (test-set fishing).
2. **Kiểm tra ổn định** (~15 phút): train lại config thắng với 3 seed khởi tạo khác nhau (tập test giữ nguyên nhờ `--init-seed`) → ra số **mean ± std** rất thuyết phục khi bảo vệ đồ án.
3. **Đóng gói** model thắng + bảng tổng hợp để tải về deploy.

**Trước khi chạy:** `Runtime` → `Change runtime type` → **T4 GPU** → Save. Tổng thời gian ~1.5–2 giờ, giữ tab mở.

In [ ]:
# 1. Kiểm tra GPU + lấy code
!nvidia-smi -L
!git clone https://github.com/thanhuy8888/-n_Systematic-Review-AI.git srai
%cd srai
!pip install -q transformers seaborn

## Bước 1 — Sweep 8 cấu hình

Quanh vùng đã biết là tốt (distil-biobert, freeze 1–2, lr 2e-5–5e-5) + một model khác họ (BioLinkBERT) làm điểm so sánh. Config `distil_lr3e5_fz2` chính là thí nghiệm A vô địch vòng 1 — làm mốc chuẩn.

In [ ]:
import subprocess, json, shutil, os
import pandas as pd

CONFIGS = {
    #  tên                      (model,                            epochs, lr,     batch, maxlen, freeze)
    "distil_lr3e5_fz2":        ("nlpie/distil-biobert",            12, "3e-5", 16, 256, 2),  # mốc chuẩn = thí nghiệm A
    "distil_lr2e5_fz2":        ("nlpie/distil-biobert",            12, "2e-5", 16, 256, 2),
    "distil_lr5e5_fz2":        ("nlpie/distil-biobert",            12, "5e-5", 16, 256, 2),
    "distil_lr3e5_fz1":        ("nlpie/distil-biobert",            12, "3e-5", 16, 256, 1),
    "distil_lr2e5_fz1":        ("nlpie/distil-biobert",            12, "2e-5", 16, 256, 1),
    "distil_lr3e5_fz2_len320": ("nlpie/distil-biobert",            12, "3e-5", 16, 320, 2),
    "distil_lr3e5_fz2_bs32":   ("nlpie/distil-biobert",            12, "3e-5", 32, 256, 2),
    "biolinkbert_lr2e5_fz6":   ("michiyasunaga/BioLinkBERT-base",  10, "2e-5", 16, 256, 6),
}

META = "sr_core/screening_model/finetuned_meta.json"
CKPT_INFO = "sr_core/screening_model/finetuned_pubmedbert_ckpt/ckpt_info.json"
MODEL_DIR = "sr_core/screening_model/finetuned_pubmedbert"
CHARTS = ["transformer_confusion_matrix.png", "transformer_roc_curve.png",
          "transformer_pr_curve.png", "transformer_probability_distribution.png"]

os.makedirs("sweep_results", exist_ok=True)
rows = []
best_val = -1.0
for name, (model, ep, lr, bs, ml, fz) in CONFIGS.items():
    print(f"\n{'='*20} {name} {'='*20}")
    cmd = ["python", "experiments/baselines/finetune_pubmedbert.py",
           "--model", model, "--epochs", str(ep), "--lr", lr,
           "--batch-size", str(bs), "--max-len", str(ml), "--freeze-layers", str(fz)]
    try:
        subprocess.run(cmd, check=True)
    except subprocess.CalledProcessError as e:
        print(f"!! {name} LỖI ({e}) — bỏ qua, chạy config tiếp theo")
        continue
    info = json.load(open(CKPT_INFO))
    meta = json.load(open(META))
    dest = f"sweep_results/{name}"
    os.makedirs(dest, exist_ok=True)
    shutil.copy(META, dest)
    shutil.copy(CKPT_INFO, dest)
    for png in CHARTS:
        shutil.copy(f"experiments/results/{png}", dest)
    rows.append({"config": name, "best_epoch": info["epoch"],
                 "val_roc_auc": round(info["val_roc_auc"], 4),
                 "val_pr_auc": round(info["val_pr_auc"], 4),
                 "test_recall": round(meta["metrics_test"]["recall"], 4),
                 "test_f1": round(meta["metrics_test"]["f1"], 4),
                 "test_roc_auc": round(meta["metrics_test"]["roc_auc"], 4),
                 "test_pr_auc": round(meta["metrics_test"]["pr_auc"], 4),
                 "test_wss95": round(meta["wss_test"], 4)})
    if info["val_roc_auc"] > best_val:
        best_val = info["val_roc_auc"]
        shutil.rmtree("sweep_results/best_model", ignore_errors=True)
        shutil.copytree(MODEL_DIR, "sweep_results/best_model")

df = pd.DataFrame(rows).sort_values("val_roc_auc", ascending=False).reset_index(drop=True)
df.to_csv("sweep_results/sweep_summary.csv", index=False)
WINNER = df.iloc[0]["config"]
print("\n===== BẢNG TỔNG HỢP (xếp theo VAL ROC-AUC — cột test chỉ để tham khảo) =====")
print(df.to_string(index=False))
print(f"\n>>> Config thắng (theo validation): {WINNER}")

## Bước 2 — Kiểm tra ổn định: 3 seeds trên config thắng

Cùng config, cùng tập test, chỉ đổi seed khởi tạo trọng số. Kết quả mean ± std cho biết con số có vững không hay chỉ là may mắn của một seed.

In [ ]:
import numpy as np

model, ep, lr, bs, ml, fz = CONFIGS[WINNER]
stab = [json.load(open(f"sweep_results/{WINNER}/finetuned_meta.json"))]  # seed 42 đã chạy ở sweep
for s in [123, 2026]:
    print(f"\n===== init-seed {s} =====")
    subprocess.run(["python", "experiments/baselines/finetune_pubmedbert.py",
                    "--model", model, "--epochs", str(ep), "--lr", lr,
                    "--batch-size", str(bs), "--max-len", str(ml),
                    "--freeze-layers", str(fz), "--init-seed", str(s)], check=True)
    stab.append(json.load(open(META)))

print(f"\n===== ỔN ĐỊNH QUA 3 SEEDS — {WINNER} (mean ± std, %) =====")
summary = {}
for k in ["precision", "recall", "f1", "roc_auc", "pr_auc"]:
    vals = [m["metrics_test"][k] for m in stab]
    summary[k] = (np.mean(vals) * 100, np.std(vals) * 100)
    print(f"  {k:10s}: {summary[k][0]:.1f} ± {summary[k][1]:.1f}")
wss_vals = [m["wss_test"] for m in stab]
print(f"  {'wss@95':10s}: {np.mean(wss_vals)*100:.1f} ± {np.std(wss_vals)*100:.1f}")
json.dump({k: {"mean": v[0], "std": v[1]} for k, v in summary.items()},
          open("sweep_results/stability_3seeds.json", "w"), indent=2)

## Bước 3 — Đóng gói model thắng để tải về

Gói gồm: trọng số model thắng (seed 42), meta + 4 biểu đồ của nó, bảng sweep đầy đủ và kết quả ổn định 3 seeds.

In [ ]:
shutil.rmtree("deploy_bundle", ignore_errors=True)
os.makedirs("deploy_bundle/sr_core/screening_model", exist_ok=True)
os.makedirs("deploy_bundle/experiments/results", exist_ok=True)
shutil.copytree("sweep_results/best_model",
                "deploy_bundle/sr_core/screening_model/finetuned_pubmedbert")
shutil.copy(f"sweep_results/{WINNER}/finetuned_meta.json",
            "deploy_bundle/sr_core/screening_model/")
for png in CHARTS:
    shutil.copy(f"sweep_results/{WINNER}/{png}", "deploy_bundle/experiments/results/")
shutil.copy("sweep_results/sweep_summary.csv", "deploy_bundle/")
shutil.copy("sweep_results/stability_3seeds.json", "deploy_bundle/")
shutil.make_archive("ket_qua_round2", "zip", "deploy_bundle")
print(f"Model thắng: {WINNER}")
from google.colab import files
files.download("ket_qua_round2.zip")

## Sau khi tải về

Gửi file `ket_qua_round2.zip` cho Claude Code trên máy local để phân tích + deploy, hoặc tự làm:
1. Giải nén, chép `finetuned_pubmedbert/` + `finetuned_meta.json` đè vào `sr_core/screening_model/`.
2. Chép 4 biểu đồ vào `experiments/results/`.
3. Số Bảng 5 = metrics trong `finetuned_meta.json`; số mean ± std trong `stability_3seeds.json` dùng cho phần bàn luận độ tin cậy.

**Chỉ thay model chính thức nếu config thắng vượt mốc chuẩn hiện tại** (thí nghiệm A: val ROC-AUC xem ở bảng sweep; test ROC-AUC 87.2%, recall 79.0%, F1 68.3%).